# Train the Jikai SetFit topic classifier

Contrastive fine-tune of `sentence-transformers/all-MiniLM-L6-v2` on the SG-tort labelled corpus (~33 train / ~8 test after 80/20 split).

SetFit is chosen because it empirically outperforms full fine-tuning below ~100 examples (Tunstall et al., 2022). One-off Colab GPU run is enough; the resulting checkpoint replaces the TF-IDF+LinearSVC baseline when `MLPipeline` is initialised with `classifier_backend='setfit'`.

In [ ]:
!pip install -q setfit datasets sentence-transformers

In [ ]:
!git clone https://github.com/gongahkia/jikai
%cd jikai

In [ ]:
import json, pathlib
from src.ml.setfit_classifier import SetFitTopicClassifier
from src.ml.data import load_data

data = load_data('data/generated/ml_bootstrap_labels.csv')
train_df = data['train']
test_df = data['test']
print(f'train={len(train_df)} test={len(test_df)}')

In [ ]:
clf = SetFitTopicClassifier()
metrics = clf.train(
    train_df['text'].tolist(),
    train_df['topic_list'].tolist(),
    num_iterations=20,
    num_epochs=1,
    batch_size=16,
)
print(metrics)

In [ ]:
eval_metrics = clf.evaluate(
    test_df['text'].tolist(),
    test_df['topic_list'].tolist(),
)
print(eval_metrics)

In [ ]:
clf.save_model('models/setfit_sg_tort')
print('Saved to models/setfit_sg_tort')

## Publishing

Optional — push the checkpoint to Hugging Face Hub so CI can pull it without committing large weights:
```python
clf.model.push_to_hub('gongahkia/jikai-setfit-sg-tort')
```